In [6]:
import json
import time
import pandas as pd
import serpapi
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
SERPAPI_KEY = os.getenv('SERPAPI_KEY')

OUTPUT_DIR = Path("../data/raw/popular_times")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = serpapi.Client(api_key=SERPAPI_KEY)


In [7]:
DAY_MAP = {
    "monday"   : 0,
    "tuesday"  : 1,
    "wednesday": 2,
    "thursday" : 3,
    "friday"   : 4,
    "saturday" : 5,
    "sunday"   : 6,
}

VENUES = pd.read_csv("../data/raw/venue/venue_malang.csv")



In [8]:
def get_place_results(keyword):
    results = client.search({
    "engine": "google_maps",
    "q": keyword,
    "type": "search"
    })

    return results.get('place_results', {})

def convert_to_24h(hour_str):
    parts = hour_str.split(" ")
    num = int(parts[0])
    period = parts[1].upper() 

    if period == "AM":
        if num == 12:
            return 0  
        return num   
    else:
        if num == 12:
            return 12 
        return num + 12 


In [9]:
rows = []

for _, venue in VENUES.iterrows():
  place_results = get_place_results(venue["venue_name"])
  popular_time = place_results.get('popular_times')
  gps_coordinate = place_results.get('gps_coordinates') or {}

  if popular_time:
    graph_results = popular_time.get('graph_results')
    
    for day_name, hours_list in graph_results.items():
      for hour_data in hours_list:
        rows.append({
          "venue_name" : venue["venue_name"],
          "lat" : gps_coordinate.get('latitude'),
          "lon" : gps_coordinate.get('longitude'),
          "zone" : venue["zone"], 
          "day_name" : day_name,
          "day_int" : DAY_MAP[day_name],
          "hour" : convert_to_24h(hour_data["time"]),
          "busyness_score" : hour_data["busyness_score"]
          })

    print(f"finished collecting data: {venue['venue_name']}")
    time.sleep(2)
  
if rows:
    df_popular_times = pd.DataFrame(rows)
    df_popular_times.to_csv("../data/raw/popular_times/popular_times_malang.csv", index=False)
else:
    print("Failed to collecting data")



    




finished collecting data: Alun-alun Kota Malang
